# 03 CMAP Age-Resolved Spatial Mapping

## Purpose
Map PanSci single-cell data (5 age groups) onto whole-body spatial transcriptomics using CMAP_py,
then run gsMap to reveal age-dependent AD genetic risk spatial dynamics.

**Paper position**: Core analysis for age-dependent AD risk figures.

### Pipeline per organ:
1. Load spatial data (Array-seq CTRL1) and PanSci single-cell data (WT only)
2. Subsample single cells per age group (3x spot count)
3. Run CMAP (Level 1-3) with filtering
4. Generate pseudo-spatial h5ad per age group
5. Run gsMap per age group → compare AD risk across ages

In [ ]:
%matplotlib inline
import omicverse as ov
ov.style(font_path='Arial')

import sys, time, os, gc, warnings
sys.path.insert(0, '../analysis/26_CMAP')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import anndata as ad
import torch
import CMAP_py

LEGEND_FS = 11; TICK_FS = 12; LABEL_FS = 13; TITLE_FS = 13; TEXT_FS = 12

def save_all_formats(fig, basepath, dpi=300):
    fig.savefig(f"{basepath}.pdf", bbox_inches="tight", dpi=dpi)
    fig.savefig(f"{basepath}.png", bbox_inches="tight", dpi=dpi)
    fig.savefig(f"{basepath}.svg", bbox_inches="tight")

BASE = '../analysis/26_gsmap'
ST_DIR = f'{BASE}/data/st/per_organ'
SC_DIR = f'{BASE}/data/pansci/per_organ'
CMAP_OUT = f'{BASE}/models/cmap_output'
os.makedirs(CMAP_OUT, exist_ok=True)
os.makedirs(f'{BASE}/figures', exist_ok=True)

print(f'CMAP_py {CMAP_py.__version__} | PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()}')

In [ ]:
# Organ mapping: PanSci name -> Array-seq h5ad name
ORGAN_MAP = {
    'Kidney':    ('GSE247719_PanSci_01_Kidney_adata.h5ad',    'Kidney_CTRL1.h5ad'),
    'Lung':      ('GSE247719_PanSci_02_Lung_adata.h5ad',      'Lung_CTRL1.h5ad'),
    'Heart':     ('GSE247719_PanSci_03_Heart_adata.h5ad',     'Heart_CTRL1.h5ad'),
    'Liver':     ('GSE247719_PanSci_04_Liver_adata.h5ad',     'Liver_CTRL1.h5ad'),
    'Muscle':    ('GSE247719_PanSci_05_Muscle_adata.h5ad',    'Muscle_CTRL1.h5ad'),
    'Stomach':   ('GSE247719_PanSci_06_Stomach_adata.h5ad',   'Stomach_CTRL1.h5ad'),
    'BAT':       ('GSE247719_PanSci_07_BAT_adata.h5ad',       'Brown_Fat_CTRL1.h5ad'),
    'iWAT':      ('GSE247719_PanSci_08_iWAT_adata.h5ad',      None),  # no matching ST
    'gWAT':      ('GSE247719_PanSci_09_gWAT_adata.h5ad',      None),  # no matching ST
    'Ileum':     ('GSE247719_PanSci_10_Ileum_adata.h5ad',     'Small_Intestine_CTRL1.h5ad'),
    'Colon':     ('GSE247719_PanSci_11_Colon_adata.h5ad',     'Colon_CTRL1.h5ad'),
    'Jejunum':   ('GSE247719_PanSci_12_Jejunum_adata.h5ad',   'Small_Intestine_CTRL1.h5ad'),
    'Duodenum':  ('GSE247719_PanSci_13_Duodenum_adata.h5ad',  'Small_Intestine_CTRL1.h5ad'),
}
# Filter to organs with matching ST data
ORGAN_MAP = {k: v for k, v in ORGAN_MAP.items() if v[1] is not None}

# Sampling config: target cells per age group = 3x spot count, capped
AGE_GROUPS = ['03_months', '06_months', '12_months', '16_months', '23_months']
SPOT_MULTIPLIER = 3
MAX_CELLS_PER_AGE = 25000  # cap for Muscle

print(f"Organs to process: {len(ORGAN_MAP)}")
for organ, (sc_f, st_f) in ORGAN_MAP.items():
    print(f"  {organ}: SC={sc_f[:30]}... ST={st_f}")

In [ ]:
def run_cmap_for_organ(organ_name, sc_file, st_file, 
                       svm_threshold=0.5, entropy_threshold=0.85, quality_threshold=0.1,
                       is_tms=False):
    """Run full CMAP pipeline for one organ. Returns sc_meta_coord or None on failure.
    is_tms: if True, use TMS data format (gene symbols as var_names, different obs columns)
    """
    
    organ_out = f'{CMAP_OUT}/{organ_name}'
    os.makedirs(organ_out, exist_ok=True)
    
    final_file = f'{organ_out}/cmap_results.csv'
    if os.path.exists(final_file):
        print(f"[SKIP] {organ_name}: already completed")
        return pd.read_csv(final_file, index_col=0)
    
    print(f"\n{'='*60}")
    print(f"[{organ_name}] Loading data...")
    print(f"{'='*60}")
    t0 = time.time()
    
    # --- Load spatial data ---
    st_adata = ad.read_h5ad(f'{ST_DIR}/{st_file}')
    n_spots = st_adata.shape[0]
    
    if n_spots > 25000:
        np.random.seed(42)
        idx = np.random.choice(n_spots, 25000, replace=False)
        st_adata = st_adata[idx].copy()
        n_spots = 25000
        print(f"  ST subsampled to {n_spots} spots")
    
    if hasattr(st_adata.X, 'toarray'):
        st_X = pd.DataFrame(st_adata.X.toarray(), index=st_adata.obs_names, columns=st_adata.var_names).T
    else:
        st_X = pd.DataFrame(st_adata.X, index=st_adata.obs_names, columns=st_adata.var_names).T
    
    spatial_location = pd.DataFrame({
        'x': st_adata.obsm['spatial'][:, 0],
        'y': st_adata.obsm['spatial'][:, 1],
        'HMRF_cluster': st_adata.obs['annotation'].values
    }, index=st_adata.obs_names)
    
    print(f"  ST: {st_X.shape[1]} spots, {st_X.shape[0]} genes")
    print(f"  Domains: {spatial_location['HMRF_cluster'].value_counts().to_dict()}")
    
    # --- Load single-cell data ---
    sc_path = f'{SC_DIR}/{sc_file}' if not is_tms else sc_file
    sc_adata = ad.read_h5ad(sc_path, backed='r')
    
    if is_tms:
        # TMS format: obs has 'age', 'cell_ontology_class', 'sex'
        wt_obs = sc_adata.obs.copy()
        celltype_col = 'cell_ontology_class' if 'cell_ontology_class' in wt_obs.columns else 'free_annotation'
        age_col = 'age'
        age_groups_local = sorted(wt_obs[age_col].dropna().unique())
    else:
        # PanSci format
        wt_mask = sc_adata.obs['Genotype'] == 'WT'
        wt_obs = sc_adata.obs[wt_mask].copy()
        celltype_col = 'Main_cell_type'
        age_col = 'Age_group'
        age_groups_local = AGE_GROUPS
    
    target_per_age = min(n_spots * SPOT_MULTIPLIER, MAX_CELLS_PER_AGE)
    sampled_indices = []
    for age in age_groups_local:
        age_obs = wt_obs[wt_obs[age_col] == age]
        n_avail = len(age_obs)
        n_sample = min(target_per_age, n_avail)
        if n_avail == 0:
            continue
        sampled = age_obs.sample(n=n_sample, random_state=42)
        sampled_indices.append(sampled.index)
        print(f"  {age}: {n_sample}/{n_avail} cells sampled")
    
    if not sampled_indices:
        print(f"  ERROR: No cells sampled. Skipping.")
        return None
    
    all_sampled = pd.Index(np.concatenate([idx.values for idx in sampled_indices]))
    sc_adata_mem = sc_adata[all_sampled].to_memory()
    
    # Convert gene names if PanSci (Ensembl -> symbol)
    if not is_tms and 'gene_name' in sc_adata_mem.var.columns:
        gene_names = sc_adata_mem.var['gene_name'].values.astype(str)
        seen = {}
        unique_names = []
        for g in gene_names:
            if g in seen:
                seen[g] += 1
                unique_names.append(f"{g}_{seen[g]}")
            else:
                seen[g] = 0
                unique_names.append(g)
        sc_adata_mem.var_names = unique_names
    
    if hasattr(sc_adata_mem.X, 'toarray'):
        sc_X = pd.DataFrame(sc_adata_mem.X.toarray(), index=sc_adata_mem.obs_names, columns=sc_adata_mem.var_names).T
    else:
        sc_X = pd.DataFrame(sc_adata_mem.X, index=sc_adata_mem.obs_names, columns=sc_adata_mem.var_names).T
    
    sc_meta = pd.DataFrame({
        'cellType': sc_adata_mem.obs[celltype_col].values,
        'Age_group': sc_adata_mem.obs[age_col].values,
    }, index=sc_adata_mem.obs_names)
    if 'Sex' in sc_adata_mem.obs.columns:
        sc_meta['Sex'] = sc_adata_mem.obs['Sex'].values
    elif 'sex' in sc_adata_mem.obs.columns:
        sc_meta['Sex'] = sc_adata_mem.obs['sex'].values
    
    print(f"  SC: {sc_X.shape[1]} cells, {sc_X.shape[0]} genes, {sc_meta['cellType'].nunique()} cell types")
    
    del sc_adata, sc_adata_mem; gc.collect()
    
    # --- Shared genes ---
    shared_genes = sorted(set(sc_X.index) & set(st_X.index))
    sc_X = sc_X.loc[shared_genes]
    st_X = st_X.loc[shared_genes]
    nonzero = (sc_X.sum(axis=1) > 0) & (st_X.sum(axis=1) > 0)
    sc_X = sc_X.loc[nonzero]; st_X = st_X.loc[nonzero]
    shared_genes = sc_X.index.tolist()
    print(f"  Shared genes: {len(shared_genes)}")
    
    if len(shared_genes) < 500:
        print(f"  ERROR: Too few shared genes. Skipping.")
        return None
    
    # --- Normalize ---
    sc_norm = CMAP_py.log_normalize(sc_X)
    st_norm = CMAP_py.log_normalize(st_X)
    spatial_genes = shared_genes
    
    # --- Level 1: Domain prediction (SVM) ---
    print(f"\n  [Level 1] Domain prediction (SVM)...")
    t1 = time.time()
    matrix = CMAP_py.data_to_transform(sc_norm, st_norm, spatial_genes, batch=True, pca_method='truncated_svd')
    
    st_cols = [c for c in matrix.columns if c in st_norm.columns]
    sc_cols = [c for c in matrix.columns if c in sc_norm.columns]
    train_set = matrix[st_cols].T.copy()
    train_set['label'] = spatial_location.loc[st_cols, 'HMRF_cluster'].values
    test_set = matrix[sc_cols].T.copy()
    
    parameters = CMAP_py.tune_parameter(train_set, test_set, cross_para=[4], verbose=False)
    best = parameters['cross_4']
    pred_st_svm, _ = CMAP_py.predict_domain(train_set, test_set, cost=best['cost'], gamma=best['gamma'], scale=True, st_svm=True)
    pred_sc_svm, prob_sc = CMAP_py.predict_domain(train_set, test_set, cost=best['cost'], gamma=best['gamma'], scale=True)
    
    max_prob = prob_sc.max(axis=1)
    sc_meta['svm_max_prob'] = max_prob
    high_conf = max_prob > svm_threshold
    frac_kept = high_conf.sum() / len(high_conf)
    if frac_kept < 0.3:
        svm_threshold_relaxed = max(0.2, np.quantile(max_prob, 0.3))
        print(f"  WARNING: Relaxing SVM threshold to {svm_threshold_relaxed:.2f} ({frac_kept:.1%} kept)")
        high_conf = max_prob > svm_threshold_relaxed
        frac_kept = high_conf.sum() / len(high_conf)
    
    n_before = len(sc_meta)
    sc_meta = sc_meta[high_conf]; pred_sc_svm = pred_sc_svm[sc_meta.index]; sc_norm = sc_norm[sc_meta.index]
    print(f"  Level 1: {n_before} -> {len(sc_meta)} cells ({frac_kept:.1%}, {time.time()-t1:.0f}s)")
    
    if len(sc_meta) < 100:
        print(f"  ERROR: Too few cells. Skipping."); return None
    
    # --- Level 2: Spot mapping ---
    print(f"  [Level 2] Spot mapping...")
    t2 = time.time()
    cell_spot_map = CMAP_py.map_cell_to_spot(
        sc_norm=sc_norm, sc_meta=sc_meta, st_norm=st_norm, spatial_location=spatial_location,
        pred_sc_svm=pred_sc_svm, pred_st_svm=pred_st_svm,
        batch=True, pca_method='pca', num_epochs=2000, para_distance=1.0, para_density=1.0,
        device='cuda' if torch.cuda.is_available() else 'cpu')
    print(f"  Level 2: {len(cell_spot_map)} cells ({time.time()-t2:.0f}s)")
    
    n_bef = len(cell_spot_map)
    csm_filt = cell_spot_map[cell_spot_map['Norm_entropy'] < entropy_threshold]
    frac_ent = len(csm_filt) / n_bef if n_bef > 0 else 0
    if frac_ent < 0.3:
        ent_rel = np.quantile(cell_spot_map['Norm_entropy'], 0.7)
        print(f"  WARNING: Relaxing entropy to {ent_rel:.2f}")
        csm_filt = cell_spot_map[cell_spot_map['Norm_entropy'] < ent_rel]
    print(f"  Entropy: {n_bef} -> {len(csm_filt)} cells")
    
    # --- Level 3: Precise location ---
    print(f"  [Level 3] Precise location...")
    t3 = time.time()
    spot_neigh_list = CMAP_py.spatial_relation_all(spatial_location.copy(), spatial_data_type='square', n_near_spot=5, dis_cut=None)
    sc_meta_coord = CMAP_py.calculate_cell_location(
        cell_spot_map=csm_filt, st_meta=spatial_location, sc_meta=sc_meta,
        sc_norm=sc_norm, st_norm=st_norm, batch=True, spot_neigh_list=spot_neigh_list, radius=1/6)
    
    sc_meta_coord = sc_meta_coord.merge(
        cell_spot_map[['Single_cell', 'Probability', 'Entropy', 'Norm_entropy']],
        left_index=True, right_on='Single_cell', how='left').set_index('Single_cell')
    sc_meta_coord['quality_score'] = sc_meta_coord['svm_max_prob'] * (1 - sc_meta_coord['Norm_entropy'])
    sc_meta_coord['organ'] = organ_name
    
    print(f"  Level 3: {len(sc_meta_coord)} cells ({time.time()-t3:.0f}s)")
    print(f"  Total: {time.time()-t0:.0f}s")
    if 'Age_group' in sc_meta_coord.columns:
        print(f"  Per-age: {sc_meta_coord['Age_group'].value_counts().sort_index().to_dict()}")
    
    sc_meta_coord.to_csv(final_file)
    cell_spot_map.to_csv(f'{organ_out}/cell_spot_map.csv', index=False)
    print(f"  Saved to {final_file}")
    return sc_meta_coord

print("CMAP pipeline function defined (with gene name conversion + TMS support).")

## Test: Run CMAP on Spleen (smallest AD-enriched organ)

In [ ]:
# But first — Spleen isn't in PanSci per-organ files. Check which organs match.
# PanSci organs: Kidney, Lung, Heart, Liver, Muscle, Stomach, BAT, iWAT, gWAT, Ileum, Colon, Jejunum, Duodenum
# Array-seq organs: Brain, Bone Marrow, Brown Fat, Colon, Heart, Kidney, Liver, Lung, Lymph Node, Muscle, 
#                   Pancreas, Skin, Small Intestine, Spleen, Stomach, Thymus
# No PanSci data for: Bone Marrow, Lymph Node, Pancreas, Skin, Spleen, Thymus
# These are exactly the immune organs with highest AD enrichment!
# So let's test with Lung instead (available in both)

print("Starting test with Lung...")
result_lung = run_cmap_for_organ('Lung', *ORGAN_MAP['Lung'])